# 02 — Comment mesurer le modèle

Je fixe les règles de comparaison avant de présenter les entraînements. Ce carnet répond à trois questions : **sur quelles données, avec quelles métriques, et comment produire un score ?**

Je relis ici des bilans sauvegardés. Les cellules sont exécutables sur CPU, sans données privées, poids, API ou GPU. Les prédictions brutes restent dans les archives locales ; les bilans agrégés sont versionnés.
[01 · Données](01_preparation_donnees.ipynb) · [Parcours](README.md) · [03 · SFT](03_sft.ipynb)


## 1. Un découpage pour apprendre, un autre pour choisir, un dernier pour mesurer

| Corpus v3 | Train : apprendre | Validation : comparer | Test : mesure finale |
|---|---:|---:|---:|
| QA | 2 098 | 262 | 262 |
| Triage | 1 888 | 238 | 226 |

Les versions FR/EN d'un cas restent dans le même split. L'audit a retiré les doublons et les cas exposés au pilote initial ; le notebook 01 explique le passage de 5 000 à 4 974 entrées.

La validation sert aux choix de modèles. Le test a été ouvert pour le **socle et le SFT successif**, avant les essais mixtes. Je ne présente donc pas le SFT mixte comme confirmé par un test indépendant. Les annotations restent des références silver, sans validation clinique.


## 2. Les métriques qui guident les décisions

| Mesure | Calcul | Pourquoi je la regarde |
|---|---|---|
| Accuracy ESI | ESI exacts / nombre d'exemples | Détail du classement 1 à 5 |
| Rappel urgent | Urgents prédits urgents / urgents de référence | Éviter de manquer les ESI 1–2 |
| Macro-F1 priorité | Moyenne des F1 des trois priorités | Évaluer aussi les classes moins fréquentes |
| Accuracy QCM | Bonnes lettres / QCM attendus | Vérifier que le triage n'efface pas la QA |
| F1 lexical QA | F1 du recouvrement de mots avec la référence | Comparer les réponses ouvertes, sans prouver leur justesse médicale |
| ESI analysable | Sorties avec un ESI extractible / sorties | Distinguer une erreur de format d'un mauvais classement |

Pour chaque classe : **précision = VP/(VP+FP)**, **rappel = VP/(VP+FN)** et **F1 = 2PR/(P+R)**. La macro-F1 moyenne les trois F1 : maximale (ESI 1–2), modérée (3), différée (4–5).

Le sous-triage critique désigne un ESI 1–2 prédit 3–5. Les sorties invalides ne doivent pas être confondues avec des décisions correctes. Le [code des métriques](../evaluation/metrics.py) fixe les dénominateurs et complète les scores par les erreurs de sur/sous-triage et l'accord bilingue.

La **loss** pilote l'apprentissage et la sélection technique des checkpoints. Les métriques de validation et la conservation de la QA guident le choix final ; ce ne sont pas la même chose.


## 3. Du cas au score : une seule chaîne

```text
exemple JSONL → consigne + demande (référence cachée)
             → tokenizer → Qwen, avec ou sans LoRA
             → réponse générée → fichier de prédictions JSONL
             → comparaison à la référence → rapport de métriques JSON
```

[inference.py](../evaluation/inference.py) charge les poids et génère sans échantillonnage. [evaluer.py](../evaluation/evaluer.py) relit les réponses et appelle les métriques : recalculer les scores ne nécessite pas de GPU. Le prompt partagé, l'EOS natif, les versions et les empreintes permettent de retrouver les conditions de chaque run ; ils ne garantissent pas une identité sur tous les moteurs.

Les générations ont été réalisées en local, puis sur Colab après des interruptions. Je conserve ce détail technique dans les journaux historiques (archives locales non publiées), sans le répéter pour chaque comparaison.


In [1]:
import json
from pathlib import Path

# Fonctionne depuis notebooks/ ou depuis la racine du dépôt.
RACINE = next(p for p in (Path.cwd(), *Path.cwd().parents)
              if (p / "serving/space/support/resultats.json").is_file())
def lire_json(relatif):
    return json.loads((RACINE / relatif).read_text(encoding="utf-8"))

resultats = lire_json("serving/space/support/resultats.json")
print("Source : bilans sauvegardés ; aucune inférence ni aucun entraînement relancé.")


Source : bilans sauvegardés ; aucune inférence ni aucun entraînement relancé.


## 4. Résultat indépendant disponible : socle contre SFT successif

Le test a été produit sur Colab par [test_final.py](../evaluation/test_final.py) selon le [protocole figé](../evaluation/protocole_test_final.json), puis récupéré dans `data/colab_archive/finalisation_20261105/`. La cellule lit uniquement le [bilan versionné](../reports/bilan_final_20261105.json). Ce résultat est présenté ici pour situer le niveau de preuve ; il ne sert pas à sélectionner le mixte présenté ensuite.


In [2]:
bilan = lire_json("reports/bilan_final_20261105.json")
print(f"{'Modèle':<20} {'Macro-F1':>10} {'ESI exact':>12} {'Rappel urgent':>15}")
for nom, r in bilan["test"].items():
    print(f"{nom:<20} {r['macro_f1_priorite']:>10.4f} {r['accuracy_esi']:>11.1%} {r['rappel_urgence_maximale']:>14.1%}")


Modèle                 Macro-F1    ESI exact   Rappel urgent
socle                    0.0556        3.1%           0.0%
sft_triage_v3            0.7858       74.8%          89.8%


## 5. Conclusion

Le SFT successif améliore nettement le triage par rapport au socle : **88 urgences retrouvées sur 98** au test. Cela ne suffit pas à choisir le modèle final : il faut aussi préserver la QA. Le carnet suivant compare les deux démarches SFT sur validation, puis explique le compromis retenu.
